# POC Model
- Initial stock ranking model within-portfolio

## Things to Test
- First Model (_Saturated_)
    - Simple temporal Train and Test 
- Second Model (_Saturated_)
    - Fit multiple models on OOT Train and test splits 
    - Fit full dataset with each model and observe ranking difference

- Variable Selection
    - Based on SHAP and temporal signal consistency

In [42]:
df = pd.read_parquet('../data/processed/modeling_dataset_28d.parquet')

In [ ]:
fred_features = [
    'sector', 'industry', 'DFF', 'DGS10', 'DGS2', 'T10Y2Y', 'CPIAUCSL',
    'CPILFESL', 'PCEPI', 'GDP', 'GDPC1', 'INDPRO', 'UNRATE', 'PAYEMS',
    'ICSA', 'UMCSENT', 'RSXFS', 'VIXCLS', 'DCOILWTICO', 'M2SL', 'TOTALSL'
]

rank_features = [col for col in df.columns if "rank" in col]

zscore_features = [col for col in df.columns if col.endswith('_zscore')]

quintile_features = [col for col in df.columns if col.endswith('_quintile')]


In [59]:
modeling_features = fred_features + rank_features + zscore_features + quintile_features

In [60]:
modeling_features

['sector',
 'industry',
 'DFF',
 'DGS10',
 'DGS2',
 'T10Y2Y',
 'CPIAUCSL',
 'CPILFESL',
 'PCEPI',
 'GDP',
 'GDPC1',
 'INDPRO',
 'UNRATE',
 'PAYEMS',
 'ICSA',
 'UMCSENT',
 'RSXFS',
 'VIXCLS',
 'DCOILWTICO',
 'M2SL',
 'TOTALSL',
 'open_rank',
 'high_rank',
 'low_rank',
 'close_rank',
 'volume_rank',
 'market_cap_rank',
 'enterprise_value_rank',
 'trailing_pe_rank',
 'forward_pe_rank',
 'peg_ratio_rank',
 'price_to_book_rank',
 'price_to_sales_rank',
 'enterprise_to_revenue_rank',
 'enterprise_to_ebitda_rank',
 'profit_margin_rank',
 'operating_margin_rank',
 'gross_margin_rank',
 'roe_rank',
 'roa_rank',
 'revenue_growth_rank',
 'earnings_growth_rank',
 'total_cash_rank',
 'total_debt_rank',
 'debt_to_equity_rank',
 'current_ratio_rank',
 'quick_ratio_rank',
 'book_value_rank',
 'revenue_per_share_rank',
 'earnings_per_share_rank',
 'dividend_rate_rank',
 'dividend_yield_rank',
 'payout_ratio_rank',
 'beta_rank',
 'shares_outstanding_rank',
 'float_shares_rank',
 'held_percent_insiders_r

## 1) Fit simple model predicting if stock will be in top 20% of returns

In [ ]:
import pandas as pd
import lightgbm as lgb

target = "target_28d"

df = df[df[target].notna()]

# Convert string columns to categorical dtype for LightGBM
categorical_features = ['sector', 'industry']
for col in categorical_features:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Time-based split
train = df[df['date'] < '2024-01-01']
test = df[df['date'] >= '2024-01-01']

# Train
model = lgb.LGBMClassifier(objective='multiclass', num_class=5)
model.fit(train[modeling_features], train[target])

# Predict and evaluate
test['pred'] = model.predict(test[modeling_features])


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.948951 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 50179
[LightGBM] [Info] Number of data points in the train set: 3926989, number of used features: 202
[LightGBM] [Info] Start training from score -1.606279
[LightGBM] [Info] Start training from score -1.611045
[LightGBM] [Info] Start training from score -1.611023
[LightGBM] [Info] Start training from score -1.611124
[LightGBM] [Info] Start training from score -1.607730


### Evaluate performance when rebalancing stocks monthly

In [ ]:
# Create year-month column for grouping
test['year_month'] = test['date'].dt.to_period('M')

# Initialize results
monthly_returns = []

for month in test['year_month'].unique():
    month_data = test[test['year_month'] == month].copy()
    
    # Get first and last date of the month
    month_dates = month_data['date'].sort_values()
    first_date = month_dates.iloc[0]
    last_date = month_dates.iloc[-1]
    
    # Get predictions at the BEGINNING of month (first date)
    first_day_data = month_data[month_data['date'] == first_date].copy()
    
    # Select stocks to long and short based on first-day predictions
    long_stocks = first_day_data[first_day_data['pred'] == 4]['ticker'].unique()
    short_stocks = first_day_data[first_day_data['pred'] == 0]['ticker'].unique()
    
    # Calculate holding period returns for each stock
    # Return = (price_end - price_start) / price_start
    long_rets = []
    for ticker in long_stocks:
        ticker_data = month_data[month_data['ticker'] == ticker].sort_values('date')
        if len(ticker_data) >= 2:
            start_price = ticker_data.iloc[0]['close']
            end_price = ticker_data.iloc[-1]['close']
            ret = (end_price - start_price) / start_price
            long_rets.append(ret)
    
    short_rets = []
    for ticker in short_stocks:
        ticker_data = month_data[month_data['ticker'] == ticker].sort_values('date')
        if len(ticker_data) >= 2:
            start_price = ticker_data.iloc[0]['close']
            end_price = ticker_data.iloc[-1]['close']
            ret = (end_price - start_price) / start_price
            short_rets.append(ret)
    
    # Equal-weighted portfolio returns
    long_ret = np.mean(long_rets) if len(long_rets) > 0 else 0
    short_ret = np.mean(short_rets) if len(short_rets) > 0 else 0
    ls_ret = long_ret - short_ret
    
    monthly_returns.append({
        'month': month,
        'long_return': long_ret,
        'short_return': short_ret,
        'ls_return': ls_ret,
        'n_long': len(long_rets),
        'n_short': len(short_rets),
        'days_held': (last_date - first_date).days
    })

# Convert to DataFrame
returns_df = pd.DataFrame(monthly_returns)

print("Monthly Long-Short Performance:")
print(returns_df)

# Summary statistics
print("\n=== Performance Summary ===")
print(f"Total Months: {len(returns_df)}")
print(f"Mean Monthly Return: {returns_df['ls_return'].mean():.4f}")
print(f"Median Monthly Return: {returns_df['ls_return'].median():.4f}")
print(f"Std Dev: {returns_df['ls_return'].std():.4f}")
print(f"Sharpe Ratio (annualized): {returns_df['ls_return'].mean() / returns_df['ls_return'].std() * np.sqrt(12):.2f}")
print(f"Win Rate: {(returns_df['ls_return'] > 0).mean():.2%}")
print(f"Max Monthly Return: {returns_df['ls_return'].max():.4f}")
print(f"Min Monthly Return: {returns_df['ls_return'].min():.4f}")

# Cumulative returns
returns_df['cumulative_return'] = (1 + returns_df['ls_return']).cumprod() - 1
print(f"\nCumulative Return: {returns_df['cumulative_return'].iloc[-1]:.2%}")

# Annualized metrics
annualized_return = returns_df['ls_return'].mean() * 12
print(f"\nAnnualized Return: {annualized_return:.2%}")
print(f"Average Holding Period: {returns_df['days_held'].mean():.1f} days")


In [ ]:
# Evaluate as long-short portfolio
for date in test['date'].unique():
    day = test[test['date'] == date]
    long_ret = day[day['pred'] == 4]['return_t+1'].mean()
    short_ret = day[day['pred'] == 0]['return_t+1'].mean()
    print(f"{date}: L/S Return = {long_ret - short_ret:.4f}")


Index(['ticker', 'date', 'open', 'high', 'low', 'close', 'volume',
       'return_t+1', 'momentum_5', 'volatility_5'],
      dtype='str')